In [1]:
from pyspark.sql import SparkSession

# Re-create the Spark session, but this time give it more RAM!
spark = SparkSession.builder \
    .appName("FlightDelayCleaning") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

# Re-load the raw data
flights_df = spark.read.csv("hdfs://namenode:9000/flight_project/raw/flights_batch.csv", header=True, inferSchema=True)
weather_df = spark.read.csv("hdfs://namenode:9000/flight_project/raw/weather_env_batch.csv", header=True, inferSchema=True)

print("Spark is running with upgraded memory!")

Spark is running with upgraded memory!


In [2]:
from pyspark.sql import SparkSession
# Initialize Spark inside the Jupyter container
spark = SparkSession.builder \
    .appName("FlightDelayCleaning") \
    .getOrCreate()

# Read the data from the namenode container!
flights_df = spark.read.csv("hdfs://namenode:9000/flight_project/raw/flights_batch.csv", header=True, inferSchema=True)
weather_df = spark.read.csv("hdfs://namenode:9000/flight_project/raw/weather_env_batch.csv", header=True, inferSchema=True)

# Show it worked
print("Flight data loaded:")
flights_df.show(3)

print("Weather data loaded:")
weather_df.show(3)


Flight data loaded:
+----+-----+------------+-----------+----------+-----------------+-----------------+------+----------------+---------------+----+--------------+-------------+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|year|month|day_of_month|day_of_week|   fl_date|op_unique_carrier|op_carrier_fl_num|origin|origin_city_name|origin_state_nm|dest|dest_city_name|dest_state_nm|crs_dep_time|dep_time|dep_delay|taxi_out|wheels_off|wheels_on|taxi_in|crs_arr_time|arr_time|arr_delay|cancelled|cancellation_code|diverted|crs_elapsed_time|actual_elapsed_time|air_time|distance|carrier_delay|weather_delay|nas_delay|security_delay|late_aircraft_delay|
+----+-----+------------+-----------+----------+-----------------+-----------------+------+----------------+---------------+----

In [3]:
from pyspark.sql.functions import col

# 1. Check how many rows we are starting with
initial_count = flights_df.count()
print(f"Initial flight count: {initial_count}")

# 2. Filter out cancelled flights (where 'cancelled' column == 1)
# We only want flights that actually happened (cancelled == 0)
active_flights_df = flights_df.filter(col("cancelled") == 0)

# 3. Drop the 'cancelled' and 'cancellation_code' columns since we don't need them anymore
clean_flights_df = active_flights_df.drop("cancelled", "cancellation_code")

# 4. Check how many rows we have left after removing the cancelled ones
final_count = clean_flights_df.count()
print(f"Final flight count: {final_count}")
print(f"Removed {initial_count - final_count} cancelled flights.")

# Verify the columns are gone
clean_flights_df.printSchema()

Initial flight count: 4714977
Final flight count: 4635225
Removed 79752 cancelled flights.
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- fl_date: date (nullable = true)
 |-- op_unique_carrier: string (nullable = true)
 |-- op_carrier_fl_num: double (nullable = true)
 |-- origin: string (nullable = true)
 |-- origin_city_name: string (nullable = true)
 |-- origin_state_nm: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- dest_city_name: string (nullable = true)
 |-- dest_state_nm: string (nullable = true)
 |-- crs_dep_time: integer (nullable = true)
 |-- dep_time: double (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- taxi_out: double (nullable = true)
 |-- wheels_off: double (nullable = true)
 |-- wheels_on: double (nullable = true)
 |-- taxi_in: double (nullable = true)
 |-- crs_arr_time: integer (nullable = true)
 |-- arr_tim

In [4]:
from pyspark.sql.functions import col, when

# Create the is_weekend feature
clean_flights_df = clean_flights_df.withColumn(
    "is_weekend",
    when(col("day_of_week").isin(1, 7), 1).otherwise(0)
)

In [5]:
# List the 5 delay columns to drop
delay_columns = [
    "carrier_delay", 
    "weather_delay", 
    "nas_delay", 
    "security_delay", 
    "late_aircraft_delay"
]

# Use the * operator to unpack the list and drop them all at once
clean_flights_df = clean_flights_df.drop(*delay_columns)

# Show the remaining columns to verify
print("Remaining columns:")
print(clean_flights_df.columns)

Remaining columns:
['year', 'month', 'day_of_month', 'day_of_week', 'fl_date', 'op_unique_carrier', 'op_carrier_fl_num', 'origin', 'origin_city_name', 'origin_state_nm', 'dest', 'dest_city_name', 'dest_state_nm', 'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off', 'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay', 'diverted', 'crs_elapsed_time', 'actual_elapsed_time', 'air_time', 'distance', 'is_weekend']


In [6]:
# List of date columns to drop
date_columns = ["year", "month", "day_of_month","day_of_week"]

# Drop them from the DataFrame
clean_flights_df = clean_flights_df.drop(*date_columns)

# Verify the updated schema
print("Columns after dropping date features:")
print(clean_flights_df.columns)

Columns after dropping date features:
['fl_date', 'op_unique_carrier', 'op_carrier_fl_num', 'origin', 'origin_city_name', 'origin_state_nm', 'dest', 'dest_city_name', 'dest_state_nm', 'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off', 'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay', 'diverted', 'crs_elapsed_time', 'actual_elapsed_time', 'air_time', 'distance', 'is_weekend']


In [7]:
leakage_and_redundant_cols = [
    # Leakage
    "dep_time", "taxi_out", "wheels_off", "wheels_on", 
    "taxi_in", "arr_time", "actual_elapsed_time", "air_time", "diverted",
    
    # Redundant
    "origin_city_name", "origin_state_nm", "dest_city_name", "dest_state_nm",
    
    # Optional: Flight number (usually too high cardinality/random for ML to learn from effectively)
    "op_carrier_fl_num" 
]

clean_flights_df = clean_flights_df.drop(*leakage_and_redundant_cols)

print("Final lean ML schema:")
clean_flights_df.printSchema()

Final lean ML schema:
root
 |-- fl_date: date (nullable = true)
 |-- op_unique_carrier: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- crs_dep_time: integer (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- crs_arr_time: integer (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- crs_elapsed_time: double (nullable = true)
 |-- distance: double (nullable = true)
 |-- is_weekend: integer (nullable = false)



In [8]:
from pyspark.sql.functions import col, when, lpad, concat, lit

# 1. EXTRACT AND ROUND THE FLIGHT TIME
# Get the hour (e.g., 1252 -> 12)
hour = (col("crs_dep_time") / 100).cast("int")
# Get the minute (e.g., 1252 -> 52)
minute = col("crs_dep_time") % 100

# If minutes >= 30, round up to the next hour. Otherwise, keep the hour.
rounded_hour = when(minute >= 30, hour + 1).otherwise(hour)

# If rounding up pushes us to 24 (midnight), reset it to 0
rounded_hour = when(rounded_hour == 24, 0).otherwise(rounded_hour)

# 2. FORMAT AS 'HH00' STRING TO MATCH WEATHER DATA
# Pad with leading zeros (so 8 becomes '08') and add '00' to the end (so '0800')
clean_flights_df = clean_flights_df.withColumn(
    "rounded_time", 
    concat(lpad(rounded_hour.cast("string"), 2, "0"), lit("00"))
)

# 3. FIX WEATHER DATA FORMAT (Just in case Spark read '0100' as the integer 100)
weather_df = weather_df.withColumn(
    "time_hhmm", 
    lpad(col("time_hhmm").cast("string"), 4, "0")
)

# Show the results to verify!
clean_flights_df.select("crs_dep_time", "rounded_time").show(5)

+------------+------------+
|crs_dep_time|rounded_time|
+------------+------------+
|        1252|        1300|
|        1015|        1000|
|        1415|        1400|
|        1650|        1700|
|        1015|        1000|
+------------+------------+
only showing top 5 rows



In [9]:
# 1. JOIN THE DATASETS
# We use a "left" join. This ensures that even if a weather reading is missing 
# for a specific hour, we don't accidentally delete the flight from our dataset!
joined_df = clean_flights_df.join(
    weather_df,
    (clean_flights_df["fl_date"] == weather_df["fl_date"]) & 
    (clean_flights_df["origin"] == weather_df["airport"]) & 
    (clean_flights_df["rounded_time"] == weather_df["time_hhmm"]),
    "left"
)

# 2. DROP THE TEMPORARY AND DUPLICATE COLUMNS
# We drop the bridge column, and the duplicate columns that came over from the weather dataset
joined_df = joined_df.drop(
    "rounded_time",          # Our temporary bridge
    weather_df["fl_date"],   # Duplicate date column from weather
    "airport",               # Duplicate of 'origin'
    "time_hhmm"              # Duplicate of our rounded time
)

# 3. VERIFY THE FINAL MASTERPIECE
print(f"Total rows after join: {joined_df.count()}")
joined_df.printSchema()

# Let's look at a single flight to see its weather attached!
joined_df.select("fl_date", "origin", "crs_dep_time", "temperature_2m", "wind_speed_10m").show(5)

Total rows after join: 4635225
root
 |-- fl_date: date (nullable = true)
 |-- op_unique_carrier: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- crs_dep_time: integer (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- crs_arr_time: integer (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- crs_elapsed_time: double (nullable = true)
 |-- distance: double (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- temperature_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- weather_code: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- pressure_msl: double (nullable = true)

+----------+------+------------+--------------+--------------+
|   fl_date|origin|crs_dep_time|temperature_2m|wind_speed_10m|
+----------+--

In [11]:
from pyspark.sql.functions import col, sum as spark_sum
final_df=joined_df
# 1.Check for missing values (Nulls) across all columns
print("Checking for missing values in the dataset:")
null_counts = final_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in final_df.columns
])

null_counts.show()

Checking for missing values in the dataset:
+-------+-----------------+------+----+------------+---------+------------+---------+----------------+--------+----------+--------------+-------------+--------+------------+--------------+--------------+----------------+------------+
|fl_date|op_unique_carrier|origin|dest|crs_dep_time|dep_delay|crs_arr_time|arr_delay|crs_elapsed_time|distance|is_weekend|temperature_2m|precipitation|snowfall|weather_code|wind_speed_10m|wind_gusts_10m|surface_pressure|pressure_msl|
+-------+-----------------+------+----+------------+---------+------------+---------+----------------+--------+----------+--------------+-------------+--------+------------+--------------+--------------+----------------+------------+
|      0|                0|     0|   0|           0|        0|           0|    13499|               0|       0|         0|             0|            0|       0|           0|             0|             0|               0|           0|
+-------+-----------

In [12]:
final_df = final_df.dropna(subset=["arr_delay"])

print(f"Final completely clean row count: {final_df.count()}")

Final completely clean row count: 4621726


In [13]:
# Select the most important numerical columns to keep the output readable
numeric_cols = [
    "arr_delay", "crs_elapsed_time", "distance", 
    "temperature_2m", "wind_speed_10m", "precipitation"
]

print("Basic Statistics (Min, Max, Mean):")
final_df.select(*numeric_cols).describe().show()

print("Full Distribution (Includes 25%, 50% Median, 75%):")
final_df.select(*numeric_cols).summary().show()

Basic Statistics (Min, Max, Mean):
+-------+-----------------+-----------------+-----------------+------------------+------------------+-------------------+
|summary|        arr_delay| crs_elapsed_time|         distance|    temperature_2m|    wind_speed_10m|      precipitation|
+-------+-----------------+-----------------+-----------------+------------------+------------------+-------------------+
|  count|          4621726|          4621726|          4621726|           4621726|           4621726|            4621726|
|   mean| 9.88138933376838|146.9784409547429| 837.628326733346| 18.31648463486687|12.405807669326721|0.13591294680812055|
| stddev|62.00342621172431|72.56954511502512|597.0684461549694|10.312455592970407| 6.891149821200812| 0.7414407649183029|
|    min|           -117.0|              6.0|             11.0|            -44.35|               0.0|                0.0|
|    max|           3803.0|            859.0|           5095.0|             49.35|         63.638584|          

In [16]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

# 1. FILTER OUTLIERS
# Keep only flights with a scheduled time of at least 30 minutes
print(f"Count before filtering outliers: {final_df.count()}")
final_df = final_df.filter(col("crs_elapsed_time") >= 30)
print(f"Count after filtering outliers: {final_df.count()}")

# 2. CONVERT TEXT TO NUMBERS (Categorical Encoding)
# The text columns we need to convert
categorical_cols = ["op_unique_carrier", "origin", "dest"]

# Create an indexer for each column that assigns a unique number to each string
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep")
    for c in categorical_cols
]

# Run all the indexers at once using a Pipeline
pipeline = Pipeline(stages=indexers)
print("Converting text categories to numbers...")
# 1. Fit the pipeline and save the fitted model to a variable
fitted_pipeline = pipeline.fit(final_df)

# 2. Transform the data for your batch layer
model_df = fitted_pipeline.transform(final_df)

# 3. SAVE THE MAPPING TO HDFS FOR THE STREAMING LAYER!
indexer_path = "hdfs://namenode:9000/flight_project/gold/string_indexer_pipeline"
fitted_pipeline.write().overwrite().save(indexer_path)
print(f"✅ StringIndexer mapping saved to {indexer_path}")
# Drop the old string columns
model_df = model_df.drop(*categorical_cols)

# Show the results!
model_df.select("op_unique_carrier_index", "origin_index", "dest_index").show(5)

Count before filtering outliers: 4621247
Count after filtering outliers: 4621247
Converting text categories to numbers...
✅ StringIndexer mapping saved to hdfs://namenode:9000/flight_project/gold/string_indexer_pipeline
+-----------------------+------------+----------+
|op_unique_carrier_index|origin_index|dest_index|
+-----------------------+------------+----------+
|                    6.0|       204.0|       1.0|
|                    0.0|        59.0|       2.0|
|                    4.0|       259.0|       0.0|
|                    4.0|       231.0|       1.0|
|                    7.0|       168.0|      49.0|
+-----------------------+------------+----------+
only showing top 5 rows



In [17]:
gold_data_path = "hdfs://namenode:9000/flight_project/gold/clean_flight_data.parquet"

print("Saving Gold data to HDFS safely...")

# Coalesce reduces the number of output files, preventing memory spikes
model_df.coalesce(1).write.mode("overwrite").parquet(gold_data_path)

print("✅ Data successfully saved to HDFS!")

Saving Gold data to HDFS safely...
✅ Data successfully saved to HDFS!
